# 05 — Point Estimation Theory
**References:** Casella & Berger (2002) Ch. 6–7 · Lehmann & Casella (1998) Ch. 1–3

## Narrative thread
```
Sufficient statistics -> Completeness -> Rao-Blackwell -> UMVUE -> Cramér-Rao bound
```

## 1. Sufficient statistics

A statistic $T(\mathbf{X})$ is **sufficient** for $\theta$ if the conditional distribution of $\mathbf{X}$ given $T(\mathbf{X})$ does not depend on $\theta$.

**Fisher-Neyman Factorization Theorem:**
$T(\mathbf{X})$ is sufficient for $\theta$ if and only if the likelihood factors as:
$$f(\mathbf{x} \mid \theta) = g(T(\mathbf{x}), \theta)\, h(\mathbf{x})$$

where $g$ depends on $\mathbf{x}$ only through $T(\mathbf{x})$, and $h$ does not depend on $\theta$.

**Minimal sufficient statistic:** A sufficient statistic $T$ is **minimal sufficient** if it is a function of every other sufficient statistic — it achieves the maximum data compression without losing information about $\theta$.

**Exponential family:** Distributions of the form $f(x\mid\theta) = h(x)\,c(\theta)\exp\!\left(\sum_j \eta_j(\theta)\,T_j(x)\right)$ have $(T_1,\ldots,T_k)$ as a sufficient statistic.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── Sufficiency: X_bar is sufficient for mu in N(mu, sigma^2 known) ───────
# The conditional distribution of (X1,...,Xn) | X_bar should not depend on mu
# Verification: sample from N(mu=5, 1) and N(mu=0, 1), condition on same X_bar

def sample_conditional_on_mean(mu, sigma, n, target_mean, n_reps=500, tol=0.05):
    """Rejection sampling: draw X ~ N(mu,sigma)^n conditioned on X_bar ≈ target_mean."""
    samples = []
    while len(samples) < n_reps:
        x = np.random.normal(mu, sigma, n)
        if abs(x.mean() - target_mean) < tol:
            samples.append(x - x.mean())  # deviations from mean
    return np.array(samples)

n = 5
target = 3.0
devs_mu5  = sample_conditional_on_mean(mu=5, sigma=1, n=n, target_mean=target)
devs_mu0  = sample_conditional_on_mean(mu=0, sigma=1, n=n, target_mean=target)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i in range(n):
    axes[0].hist(devs_mu5[:, i], bins=30, alpha=0.5, density=True, label=f'$X_{i+1}-\\bar X$')
axes[0].set_title(f'$\\mu=5$: Conditional distribution of $X_i - \\bar X$ given $\\bar X \\approx {target}$')

for i in range(n):
    axes[1].hist(devs_mu0[:, i], bins=30, alpha=0.5, density=True, label=f'$X_{i+1}-\\bar X$')
axes[1].set_title(f'$\\mu=0$: Conditional distribution of $X_i - \\bar X$ given $\\bar X \\approx {target}$')

plt.suptitle('Sufficiency: conditional distribution does not depend on μ\n'
             'Both plots should look the same — $\\bar{X}$ captures all info about μ',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Completeness and the Rao-Blackwell theorem

A statistic $T(\mathbf{X})$ is **complete** for $\theta$ if:
$$E_\theta[g(T)] = 0 \;\forall\,\theta \implies P_\theta(g(T) = 0) = 1 \;\forall\,\theta$$

Informally: no nontrivial function of $T$ has expectation zero for all $\theta$.

**Rao-Blackwell Theorem:** Let $W$ be any unbiased estimator of $\theta$, and $T$ a sufficient statistic. Define $\tilde{W} = E[W \mid T]$. Then:
1. $\tilde{W}$ is a valid statistic (does not depend on $\theta$)
2. $\tilde{W}$ is unbiased: $E[\tilde{W}] = \theta$
3. $\text{Var}(\tilde{W}) \leq \text{Var}(W)$ — conditioning on $T$ never increases variance

**Lehmann-Scheffé Theorem:** If $T$ is a **complete sufficient** statistic and $\tilde{W} = E[W \mid T]$ is unbiased, then $\tilde{W}$ is the **UMVUE** (Uniformly Minimum Variance Unbiased Estimator) — it has the smallest variance among all unbiased estimators, for all $\theta$.

In [ ]:
# ── Rao-Blackwell: improve estimator of P(X=0) for Poisson(lambda) ────────
# X1,...,Xn ~ Poisson(lambda),  T = sum(Xi) is complete sufficient
# Naive estimator:  W = 1(X1=0) = e^{-lambda}  (unbiased)
# Rao-Blackwell:    E[W | T=t] = ((n-1)/n)^t  (UMVUE)

lam_true = 2.0
n_data   = 10
n_reps   = 50_000
theta_true = np.exp(-lam_true)  # true P(X=0)

naive_ests = []
rb_ests    = []

for _ in range(n_reps):
    x = np.random.poisson(lam_true, n_data)
    t = x.sum()
    naive_ests.append(int(x[0] == 0))         # 1(X1=0)
    rb_ests.append(((n_data - 1) / n_data)**t) # UMVUE via Rao-Blackwell

naive_ests = np.array(naive_ests)
rb_ests    = np.array(rb_ests)

print('=== Rao-Blackwell improvement ===')
print(f'  True P(X=0) = e^(-lambda) = {theta_true:.6f}')
print()
print(f'  Naive W=1(X1=0):')
print(f'    Bias:     {naive_ests.mean() - theta_true:.6f}')
print(f'    Variance: {naive_ests.var():.6f}')
print(f'    MSE:      {((naive_ests - theta_true)**2).mean():.6f}')
print()
print(f'  UMVUE via Rao-Blackwell E[1(X1=0)|T]:')
print(f'    Bias:     {rb_ests.mean() - theta_true:.6f}')
print(f'    Variance: {rb_ests.var():.6f}')
print(f'    MSE:      {((rb_ests - theta_true)**2).mean():.6f}')
print(f'\n  Variance reduction: {(1 - rb_ests.var()/naive_ests.var())*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, est, name in zip(axes, [naive_ests, rb_ests], ['Naive W = 1(X₁=0)', 'UMVUE (Rao-Blackwell)']):
    ax.hist(est, bins=50, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
    ax.axvline(theta_true, color='#f72585', lw=2, linestyle='--', label=f'True $e^{{-\\lambda}}$={theta_true:.3f}')
    ax.axvline(est.mean(),  color='#06d6a0', lw=2, label=f'Mean={est.mean():.4f}')
    ax.set_title(f'{name}\nVar={est.var():.5f}')
    ax.legend(fontsize=8)

plt.suptitle('Rao-Blackwell Theorem: UMVUE has lower variance than naive unbiased estimator',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. The Cramér-Rao lower bound

For any unbiased estimator $W$ of $g(\theta)$ (under regularity conditions):

$$\text{Var}_\theta(W) \geq \frac{[g'(\theta)]^2}{I_n(\theta)}$$

where the **Fisher information** is:
$$I_n(\theta) = n\,I(\theta), \qquad I(\theta) = E_\theta\!\left[\left(\frac{\partial}{\partial\theta}\log f(X\mid\theta)\right)^2\right] = -E_\theta\!\left[\frac{\partial^2}{\partial\theta^2}\log f(X\mid\theta)\right]$$

The **score function** is $s(\mathbf{x};\theta) = \frac{\partial}{\partial\theta}\log f(\mathbf{x}\mid\theta)$, which has $E[s] = 0$ under the model.

**Efficiency:** An estimator achieving the CRB is called **efficient**. The MLE is asymptotically efficient (Notebook 06).

**Multiparameter CRB:** For vector $\boldsymbol{\theta}$, the Fisher information matrix $I(\boldsymbol{\theta})_{ij} = \text{Cov}(s_i, s_j)$, and for any unbiased estimator $\hat{\theta}$:
$$\text{Cov}(\hat{\boldsymbol{\theta}}) \geq I(\boldsymbol{\theta})^{-1} \quad \text{(matrix inequality)}$$

In [ ]:
# ── Cramér-Rao bound for Normal mean: I(mu) = n/sigma^2 ──────────────────
# X1,...,Xn ~ N(mu, sigma^2),  sigma known
# UMVUE = X_bar,  Var(X_bar) = sigma^2/n = 1/I_n(mu)  => efficient!

sigma = 2.0
ns    = np.arange(5, 201, 5)
n_reps = 20_000

emp_var_xbar = []
crb          = []

for n in ns:
    means = np.random.normal(0, sigma, size=(n_reps, n)).mean(axis=1)
    emp_var_xbar.append(means.var())
    crb.append(sigma**2 / n)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ns, crb, 'r-', lw=2.5, label='CRB = $\\sigma^2/n$')
ax.plot(ns, emp_var_xbar, 'o', color='#4361ee', markersize=5, alpha=0.7, label='Empirical Var($\\bar{X}_n$)')
ax.set_xlabel('Sample size $n$')
ax.set_ylabel('Variance')
ax.set_title('Cramér-Rao Bound: $\\bar{X}$ achieves the bound (is efficient)\n'
             '$\\text{Var}(\\bar{X}_n) = \\sigma^2/n = 1/I_n(\\mu)$')
ax.legend()
plt.tight_layout()
plt.show()

# ── Fisher information for Bernoulli(p) ──────────────────────────────────
ps = np.linspace(0.01, 0.99, 300)
fisher_bernoulli = 1 / (ps * (1 - ps))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ps, fisher_bernoulli, color='#4361ee', lw=2.5)
ax.set_xlabel('p'); ax.set_ylabel('$I(p) = 1/[p(1-p)]$')
ax.set_title('Fisher Information for Bernoulli(p)\n'
             'Peaks at p=0 or p=1 (extreme probabilities are most informative)')
ax.set_ylim(0, 100)
ax.axvline(0.5, color='#f72585', lw=1.5, linestyle='--', label='min at p=0.5')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Key takeaways

| Concept | Statement |
|---|---|
| Sufficient statistic | $T$ captures all information about $\theta$ in the data |
| Factorization theorem | $f(\mathbf{x}\mid\theta) = g(T,\theta)\,h(\mathbf{x})$ |
| Rao-Blackwell | Conditioning on sufficient stat never increases variance |
| UMVUE | Best unbiased estimator — achieved via complete sufficient stat |
| Cramér-Rao bound | $\text{Var}(W) \geq 1/I_n(\theta)$ — theoretical floor on variance |
| Efficiency | Estimator achieving CRB; MLE is asymptotically efficient |

**Next:** notebook 06 — MLE theory: existence, consistency, asymptotic normality, and information.